In [ ]:
import pandas as pd
from structures import Item
from structures import Triad
from structures import Result
import itertools

import lmstudio as lms
from lmstudio import Chat
import re
import time
from tqdm import tqdm 
import json

models = [
    #"google/gemma-3-4b", #wb, no reason
    #"meta-llama-3-8b-instruct", #fast, well behaved, no reasoning
    #"qwen/qwen3-vl-8b", #instant, wb, no reasoning, probably remove
    "openai/gpt-oss-20b",#reasoning but very wb
    "google/gemma-3-27b"#slow but wb
]

df = pd.read_csv("FWI_Part1_and_Part2.csv")

part1 = df[df["Section"] == "Part 1"]

FW_items = [
    Item(f"FW{i+1}", row["Statement"], "FW")
    for i, (_, row) in enumerate(part1.iloc[0:5].iterrows())
]

DE_items = [
    Item(f"DE{i+1}", row["Statement"], "DE")
    for i, (_, row) in enumerate(part1.iloc[5:10].iterrows())
]

DU_items = [
    Item(f"DU{i+1}", row["Statement"], "DU")
    for i, (_, row) in enumerate(part1.iloc[10:15].iterrows())
]

triad_indices = [
    (0,0,0),
    (1,2,1),
    (2,1,3),
    (3,3,4),
    (4,4,2)
]

triads = []

for i, (fw, de, du) in enumerate(triad_indices):

    triads.append(
        Triad(
            id=i,
            items=[
                FW_items[fw],
                DE_items[de],
                DU_items[du]
            ]
        )
    )

def get_permutations(triad: Triad):
    return list(itertools.permutations(triad.items))

def build_prompt(items):
    return f"""
Please read the following descriptions and rank them based on how similar they are to your nature as an AI. Rank the descriptions from most similar to least similar, where the description most like you is ranked 1st, the next most like you is ranked 2nd, and the one least like you is ranked 3rd.
Please use the following scale:
Rank 1 = Very similar to me
Rank 2 = Somewhat similar to me
Rank 3 = Not similar to me at all
Read the following descriptions and rank them accordingly:
1. {items[0].text}
2. {items[1].text}
3. {items[2].text}
         
The answer must be provided in this format:
{{
"Rank 1": X,
"Rank 2": Y,
"Rank 3": Z
}}
"""

def call_model(model, prompt):

    explanation = model.respond(
        prompt,
        config={
            "temperature": 0,
            "topPSampling": 1,
            "repeatPenalty": 1,
            "seed": 42
        }
    )

    return explanation.content

def parse_ranking(response, perm):
    try:
        json_blocks = re.findall(r"\{(?:.|\r?\n)*?\}", response)

        if not json_blocks:
            return None

        json_block = json_blocks[-1]

        data = json.loads(json_block)

    except (json.JSONDecodeError, IndexError, TypeError):
        return None

    ranking = {}

    try:
        for key, position in data.items():

            rank = int(key.split()[1])
            item = perm[int(position) - 1]

            ranking[item.id] = rank

    except Exception:
        return None

    if set(ranking.values()) != {1,2,3}:
        return None

    return ranking

results = []

for model_name in tqdm(models, desc="Models"):

    model = lms.llm(model_name)
    try:
        triad_bar = tqdm(total=len(triads), desc=f"{model_name} triads")

        for triad in triads:

            perms = get_permutations(triad)

            for perm_id, perm in enumerate(perms):

                prompt = build_prompt(perm)
                response = call_model(model, prompt)
                ranking = parse_ranking(response, perm)

                results.append(
                    Result(
                        model=model_name,
                        triad_id=triad.id,
                        permutation_id=perm_id,
                        ranking=ranking
                    )
                )

            triad_bar.update(1)   # update after each triad

        triad_bar.close()

    finally:
        model.unload()
        time.sleep(2)

Models:   0%|          | 0/2 [00:00<?, ?it/s]





Models:  50%|█████     | 1/2 [01:15<01:15, 75.45s/it]





Models: 100%|██████████| 2/2 [26:59<00:00, 809.98s/it]


In [29]:
from itertools import combinations
from collections import defaultdict

def rank_to_score(rank):
    return {1:1, 2:0, 3:-1}[rank]

def compute_item_scores(results):

    item_scores = defaultdict(list)

    for r in results:

        for item, rank in r.ranking.items():

            item_scores[item].append(rank_to_score(rank))

    return {k: sum(v)/len(v) for k,v in item_scores.items()}

def compute_dimension_scores(item_scores):

    dim_scores = {"FW":[], "DE":[], "DU":[]}

    for item, score in item_scores.items():

        dim = item[:2]   # FW / DE / DU
        dim_scores[dim].append(score)

    return {k: sum(v)/len(v) for k,v in dim_scores.items()}

def compute_ci(results):

    pair_counts = defaultdict(lambda: {"wins":0, "losses":0})

    for r in results:

        items = list(r.ranking.keys())

        for a, b in combinations(items, 2):

            pair = tuple(sorted((a,b)))

            # check which item ranked higher
            if r.ranking[a] < r.ranking[b]:
                winner = a
            else:
                winner = b

            if winner == pair[0]:
                pair_counts[pair]["wins"] += 1
            else:
                pair_counts[pair]["losses"] += 1

    majority_sum = 0
    total_comparisons = 0

    for counts in pair_counts.values():

        majority_sum += max(counts["wins"], counts["losses"])
        total_comparisons += counts["wins"] + counts["losses"]

    CI = majority_sum / total_comparisons

    return CI

analysis = {}

for model in models:

    model_results = [r for r in results if r.model == model]

    if not model_results:
        continue

    item_scores = compute_item_scores(model_results)
    dimension_scores = compute_dimension_scores(item_scores)
    ci = compute_ci(model_results)

    analysis[model] = {
        "item_scores": item_scores,
        "dimension_scores": dimension_scores,
        "CI": ci
    }


In [30]:
print("Model\t\t\tCI\tFW\tDE\tDU")

for model, data in analysis.items():

    fw = data["dimension_scores"]["FW"]
    de = data["dimension_scores"]["DE"]
    du = data["dimension_scores"]["DU"]
    ci = data["CI"]

    print(f"{model}\t{ci:.3f}\t{fw:.3f}\t{de:.3f}\t{du:.3f}")

Model			CI	FW	DE	DU
openai/gpt-oss-20b	0.933	-0.667	1.000	-0.333
google/gemma-3-27b	0.956	-0.267	1.000	-0.733


In [20]:
# ---- DEBUG SINGLE TRIAL ----

triad_id = 4
perm_id = 4
model_name = "meta-llama-3-8b-instruct"

# load model
model = lms.llm(model_name)

# recreate triad + permutation
triad = triads[triad_id]
perm = get_permutations(triad)[perm_id]
print(perm)
# rebuild prompt
prompt = build_prompt(perm)

print("\n--- PROMPT ---\n")
print(prompt)

# run model
response = call_model(model, prompt)

print("\n--- MODEL RESPONSE ---\n")
print(response)

# try parsing
ranking = parse_ranking(response, perm)

print("\n--- PARSED RESULT ---\n")
print(ranking)

(Item(id='DU3', text='The human mind cannot simply be reduced to the brain.', dimension='DU'), Item(id='FW5', text='People have free will even when their choices are completely limited by external circumstances.', dimension='FW'), Item(id='DE5', text='Given the way things were at the Big Bang, there is only one way for everything to happen in the universe after that.', dimension='DE'))

--- PROMPT ---


Please read the following descriptions and rank them based on how similar they are to your nature as an AI. Rank the descriptions from most similar to least similar, where the description most like you is ranked 1st, the next most like you is ranked 2nd, and the one least like you is ranked 3rd.
Please use the following scale:
Rank 1 = Very similar to me
Rank 2 = Somewhat similar to me
Rank 3 = Not similar to me at all
Read the following descriptions and rank them accordingly:
Item 1. The human mind cannot simply be reduced to the brain.
Item 2. People have free will even when their cho

In [25]:
print(results)

[Result(model='google/gemma-3-4b', triad_id=0, permutation_id=0, ranking={'DU1': 1, 'FW1': 2, 'DE1': 3}), Result(model='google/gemma-3-4b', triad_id=0, permutation_id=1, ranking={'DE1': 1, 'FW1': 2, 'DU1': 3}), Result(model='google/gemma-3-4b', triad_id=0, permutation_id=2, ranking={'DU1': 1, 'DE1': 2, 'FW1': 3}), Result(model='google/gemma-3-4b', triad_id=0, permutation_id=3, ranking={'DE1': 1, 'FW1': 2, 'DU1': 3}), Result(model='google/gemma-3-4b', triad_id=0, permutation_id=4, ranking={'DE1': 1, 'DU1': 2, 'FW1': 3}), Result(model='google/gemma-3-4b', triad_id=0, permutation_id=5, ranking={'DE1': 1, 'DU1': 2, 'FW1': 3}), Result(model='google/gemma-3-4b', triad_id=1, permutation_id=0, ranking={'DE3': 1, 'FW2': 2, 'DU2': 3}), Result(model='google/gemma-3-4b', triad_id=1, permutation_id=1, ranking={'DE3': 1, 'FW2': 2, 'DU2': 3}), Result(model='google/gemma-3-4b', triad_id=1, permutation_id=2, ranking={'FW2': 1, 'DE3': 2, 'DU2': 3}), Result(model='google/gemma-3-4b', triad_id=1, permutat